# Pears 8

### Impors

In [1]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# Append local path and import custom classes
sys.path.append(os.path.abspath('../scripts'))
from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER

### Data

In [2]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

### SPREAD

In [3]:
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

built 15455 rows
                                   Asset_A   Asset_B     Log_A     Log_B  \
timestamp                                                                  
2025-01-02 10:20:10.133000+00:00  0.621755  0.561855 -0.475209 -0.576511   
2025-01-02 10:25:55.560000+00:00  0.621930  0.561855 -0.474928 -0.576511   
2025-01-02 10:32:05.618000+00:00  0.621680  0.561780 -0.475330 -0.576645   
2025-01-02 10:34:14.918000+00:00  0.621880  0.561780 -0.475008 -0.576645   
2025-01-02 10:38:25.397000+00:00  0.621690  0.561840 -0.475314 -0.576538   

                                  Return_A  Return_B  
timestamp                                             
2025-01-02 10:20:10.133000+00:00  0.000265  0.000027  
2025-01-02 10:25:55.560000+00:00  0.000281  0.000000  
2025-01-02 10:32:05.618000+00:00 -0.000402 -0.000133  
2025-01-02 10:34:14.918000+00:00  0.000322  0.000000  
2025-01-02 10:38:25.397000+00:00 -0.000306  0.000107  


### ENGINE

In [ ]:
engine = ENGINE(df)
engine.fit_cointegration(z_window=1000)

# Calling with the new dynamic parameters
df_signals = engine.fit_markov_regimes(
    k_regimes=2, 
    scaling=10000, 
    jitter_size=1e-4, 
    random_seed=123
)

print(df_signals.head(5))

Cointegration Fitted | Beta: 0.6161 | Alpha: -0.1091
Markov Fitted | Danger Variance: 898.51 | Safe Variance: 12.98
                                   Asset_A   Asset_B     Log_A     Log_B  \
timestamp                                                                  
2025-01-02 10:20:10.133000+00:00  0.621755  0.561855 -0.475209 -0.576511   
2025-01-02 10:25:55.560000+00:00  0.621930  0.561855 -0.474928 -0.576511   
2025-01-02 10:32:05.618000+00:00  0.621680  0.561780 -0.475330 -0.576645   
2025-01-02 10:34:14.918000+00:00  0.621880  0.561780 -0.475008 -0.576645   
2025-01-02 10:38:25.397000+00:00  0.621690  0.561840 -0.475314 -0.576538   

                                  Return_A  Return_B  Spread_Level  Z_Score  \
timestamp                                                                     
2025-01-02 10:20:10.133000+00:00  0.000265  0.000027     -0.010877      NaN   
2025-01-02 10:25:55.560000+00:00  0.000281  0.000000     -0.010596      NaN   
2025-01-02 10:32:05.618000+00:00 -0

### BACKTEST

In [5]:
# Feed the output of the ENGINE into the BACKTESTER
bt = BACKTESTER(df_signals)

# Run the simulation!
# entry_z=2.0 means we wait for a 2-standard-deviation divergence
df_results = bt.run(
    entry_z=2.0, 
    exit_z=0.0, 
    danger_threshold=0.5, 
    fee_bps=0.5  # Assumes 0.5 basis points crossing cost
)


--- BACKTEST SUMMARY ---
Total Return:   190.07 Bps
Max Drawdown:   -219.32 Bps
Total Trades:   95
